# Ch.2  データを「見る」

【講師用完全版】

| 項目 | 内容 |
|------|------|
| 使うデータ | ワインデータ（Ch.1 と同じ）178件・13特徴量・3品種 |
| 演習時間 | 40分（問3まで完了で合格） |
| ゴール | グラフから「この変数が品種を決めている」という仮説を立てる |

---
### 📌 講師メモ：時間配分と時間が押した場合の対応

| 区分 | 内容 | 目安時間 |
|------|------|---------|
| 座学（concept_ch2） | EDA の目的・グラフ種類・各グラフの読み方 | 25分 |
| 演習（STEP1〜問1） | ヒストグラム・箱ひげ図・散布図 | 25分 |
| 演習（問2〜問3） | ヒートマップ・仮説の言語化（AI禁止） | 15分 |
| 問4（発展） | pairplot | 余り次第 |

**時間が押した場合：**
- 問3（仮説の言語化）を「Ch.3 特徴量重要度の後に記入」に延期してもよい
- 問4（pairplot）は最初から発展扱いと案内する
- **問2の相関ヒートマップまで完了していれば Ch.3 に進んで問題ない**

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇のグラフを描きたい / 〇〇を可視化したい
【使うライブラリ】matplotlib / seaborn
【データの形】df という DataFrame、178行×14列（数値13列 + 品種名列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

「グラフを描いて」ではなく、
「df の alcohol 列と proline 列を散布図で描きたい。品種名ごとに色を変えたい。どう書くか」のように、
**列名・グラフの種類・色分けの条件** をセットで伝えましょう。

---
## 準備  ライブラリとデータを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = ["wine_" + str(t) for t in wine.target]
print(f"行数: {len(df)}  列数: {len(df.columns)}")

---
## STEP 1  データの「分布の形」を見る

なぜ分布を確認するのか：
「平均値」は一点の代表値に過ぎません。
値がどの範囲に集まっているか・偏りがあるか・外れ値がないかは、グラフで見るとわかります。

---

Q1-1：アルコール度数（alcohol）はどんな分布をしていますか？

ヒストグラムを描いて、値の広がり方・集中具合を確認してください。

💡 ヒント Copilot への相談例：
「pandas の DataFrame の特定の列をヒストグラムで表示したい。タイトルと軸ラベルも付けたい」

In [ ]:
# alcohol のヒストグラムを描いて分布の形を確認する
plt.hist(df["alcohol"], bins=20)
plt.title("alcohol の分布")
plt.xlabel("アルコール度数")
plt.ylabel("件数")
plt.show()

---
### STEP 1 の気づき（AI 禁止：グラフを見てメモ）

Q：ヒストグラムの山はいくつありましたか？その形から何が想像できますか？

>

✅ （模範解答）2〜3 つの山が見える。これは複数の品種が混ざっているため。1品種のデータであれば通常 1つの山になる。

📌 講師メモ：「なぜ山が複数あると思う？」と品種の混在を考えさせ、STEP 2 の品種別グラフへの動機付けをする。

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 分布の形 | 11〜15 の範囲に分布。やや 2〜3 つの山がある（品種の混在を示す） |
| 中心 | 12〜13 あたりに集中 |
| 外れ値 | 大きな外れ値はなく、比較的きれいな分布 |

📌 ポイント 「2つの山」や「右裾が長い」分布は、複数のグループが混ざっているサインかもしれない。
品種別に分けてグラフを描くと、この山の理由がわかることがある（問1で確認する）。

📌 講師メモ: 「この山は何を表していると思う？」と品種の混在を考えさせる。

---
## STEP 2  品種ごとに「差」を比べる

なぜ品種別で比べるのか：
Ch.1 の集計で「品種によって平均が違う」ことがわかりました。
箱ひげ図を使うと「中央値の違い」「ばらつきの大きさ」「外れ値の有無」をグループ別に一度に確認できます。

---

Q2-1：品種ごとに alcohol の分布を比べてみましょう

3品種（wine_0・wine_1・wine_2）ごとにアルコール度数の分布を比べてください。

💡 ヒント Copilot への相談例：
「pandas の DataFrame でグループ別に列の箱ひげ図を描きたい」

In [ ]:
# 品種ごとの alcohol の分布を箱ひげ図で比較する
df.boxplot(column="alcohol", by="品種名")
plt.suptitle("")
plt.title("品種別 alcohol の分布")
plt.ylabel("アルコール度数")
plt.show()

---
### STEP 2 の気づき（AI 禁止：グラフを見てメモ）

Q：3品種のうち、最もアルコール度数に差があるペアはどれですか？また、見分けにくそうな品種ペアはどれですか？

>

✅ （模範解答）wine_0（中央値 約 13.7）と wine_1（中央値 約 12.3）の差が最も大きく、見分けやすい。wine_0 と wine_2 は中央値が近く、alcohol だけでは区別が難しい。

📌 講師メモ：「wine_0 と wine_2 を分けるにはどの変数が必要か？」と問い返し、散布図（問1）への動機付けをする。

---
### 📊 出力結果の解説

| 品種 | 中央値（目安） | 特徴 |
|------|-------------|------|
| wine_0 | 約 13.7 | 最も高く、ばらつきも比較的小さい |
| wine_1 | 約 12.3 | 最も低い |
| wine_2 | 約 13.1 | 中間だが wine_0 と重なりが多い |

📌 ポイント wine_0 と wine_1 は中央値で 1.4 度差があり、ほぼ重ならない → 分類しやすい。
wine_0 と wine_2 は重なりが多い → alcohol だけでは区別が難しいかもしれない。
「重なりが少ない変数 = 機械学習で効く変数」の目安になる。

📌 講師メモ: 「wine_0 と wine_2 を分けるには何の変数が使えそうか？」と問い返す。

---
## 問1  2つの変数の「関係」をグラフで確認する ★☆☆

なぜ散布図を使うのか：
箱ひげ図は 1 変数。散布図は 2 変数の「組み合わせ」で見ます。
「alcohol が高いと proline も高い品種がある」という関係は、散布図で初めて見えます。

---

Q：alcohol（X軸）と proline（Y軸）の散布図を、品種別に色を分けて描いてみましょう

3品種が色分けされた散布図を描いて、品種がグラフ上で「分離しているか」を確認してください。

📌 なぜ alcohol と proline？：Ch.1 の groupby で wine_0 の proline が他品種の約 2 倍あり、alcohol も品種間の差が大きかった変数です。「差が大きい 2 変数の組み合わせ」を散布図で見ます。

💡 ヒント Copilot への相談例：
「matplotlib で散布図を描きたい。品種名列ごとに色を変えて、凡例も表示したい」

In [ ]:
# 品種別に色を分けた散布図を描いて、品種がグラフ上で分離しているか確認する
for name in df["品種名"].unique():
    group = df[df["品種名"] == name]
    plt.scatter(group["alcohol"], group["proline"], label=name, alpha=0.7)
plt.xlabel("alcohol（アルコール度数）")
plt.ylabel("proline（プロリン）")
plt.title("alcohol vs proline（品種別）")
plt.legend()
plt.show()

---
### 問1 の気づき（AI 禁止：1行でOK）

Q：散布図で3品種は分かれていましたか？どのくらい分離していましたか？

>

✅ （模範解答）wine_0（右上・alcohol高・proline高）と wine_1（左下・両方低）はほぼ完全に分かれており、wine_2 が中間に位置する。2変数の組み合わせで3品種がほぼ識別できる。

📌 講師メモ：「wine_0 と wine_1 は右上と左下に完全に分かれています。これが "2変数の組み合わせで品種を区別できる" ということです。Ch.3 の機械学習はこの分離を自動的に学習します」と口頭補足する。

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| wine_0（右上） | alcohol 高 + proline 高 → 右上にまとまる |
| wine_1（左下） | alcohol 低 + proline 低 → 左下にまとまる |
| wine_2（中間） | 中間付近に分布 |

📌 ポイント 3品種が比較的きれいに分離している = この 2 変数の組み合わせで品種を区別できる可能性がある。
これが Ch.3 での機械学習（RandomForest）への「仮説」となる。

📌 講師メモ: 「この散布図を見ると、wine_0 と wine_1 は間違えにくそうですね」と声がけ。

✅ （模範解答）wine_0 は右上（alcohol 高・proline 高）、wine_1 は左下（両方低）にまとまっており、
2 変数の組み合わせで 3 品種がほぼ分離できることが確認できる。

---
## 問2  全変数の「関係の強さ」を一度に確認する ★★☆

なぜ相関ヒートマップを使うのか：
13 列もある変数を 1 つずつ散布図で確認するのは時間がかかります。
ヒートマップを使うと「強く相関する変数ペア」を一度に発見できます。

---

📌 相関係数とは：2つの変数が「一緒に増える・一緒に減る」関係の強さを **-1〜+1** の数値で表したもの。  
- **+1 に近い**：一方が増えるともう一方も増える（正の相関）  
- **-1 に近い**：一方が増えるともう一方は減る（負の相関）  
- **0 に近い**：ほぼ無関係  
目安：**絶対値が 0.7 以上**なら「強い相関あり」と判断します。

---

Q：全数値列の相関係数をヒートマップで表示してみましょう

ヒートマップを描いて、**赤いマス（+0.7 以上）や青いマス（-0.7 以下）** を 1 つ見つけて、「それは何と何の変数ペアか」を確認してください。

💡 ヒント Copilot への相談例：
「seaborn で DataFrame の数値列の相関行列をヒートマップで表示したい。数値も表示したい」

In [ ]:
# 数値列だけを抜き出して相関行列を計算し、ヒートマップで表示する
corr = df.select_dtypes(include="number").corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm")
plt.title("相関ヒートマップ")
plt.tight_layout()
plt.show()

---
### 問2 の気づき（AI 禁止：1行でOK）

Q：最も強い相関があった変数ペアはどれですか？（ヒートマップの赤いマスを見て確認）

>

✅ （模範解答）flavanoids と total_phenols の相関が約 0.9 で最も強い。どちらもポリフェノール系の成分で化学的に関連しているため。

📌 講師メモ：「相関が強い変数は "同じ情報を測っている" 可能性がある。機械学習で両方入れても効果が重複する場合がある」と口頭で補足する。

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 赤いマス（+1.0 に近い） | 強い正の相関 → flavanoids と total_phenols（≈ 0.9）など |
| 青いマス（-1.0 に近い） | 強い負の相関 → color_intensity と hue（≈ -0.5）など |
| 淡い色のマス | 弱い相関（ほぼ無関係） |

📌 ポイント 相関が高い変数は「同じ情報を持っている」可能性がある。
flavanoids と total_phenols が強く相関 → 機械学習で両方入れても意味が薄いかもしれない。
「重要な変数を絞り込む」ための手がかりになる。

📌 講師メモ: 「どの変数ペアが一番強く関係していますか？」と問い返す。

✅ （模範解答）flavanoids と total_phenols の相関が 0.9 程度で最も強い。
これらは「ポリフェノール系」という同じ化学的性質を持つため、どちらか一方が代表できる可能性がある。

---
## 問3  グラフから「仮説」を言葉にする ★★☆（AI 禁止・最重要）

実務でのイメージ：
グラフは「見るもの」ではなく「仮説を立てるツール」です。
「この変数が品種を決めているのでは？」という仮説を言葉にする練習をします。

---

⛔ 注意 AI は使わないこと。グラフを見て、自分の言葉で書きましょう。

💡 書き方の例：「〇〇のグラフを見ると、〇〇（品種）は〇〇（変数）が他品種より大きい／小さい。そのため〇〇が品種を分けやすい変数と考える。」

【仮説1】品種を最も分けやすそうな変数はどれですか？その理由は？

>

【仮説2】機械学習のモデル（Ch.3）で特に重要になりそうな変数を 2〜3 個予想してください。

>

【確認】Ch.3 の特徴量重要度で予想と合っていたかを確認しましょう（Ch.3 後に記入）

>

---
### 📌 講師メモ（問3 の模範仮説）

✅ （模範解答）
仮説1：proline が最も品種を分けやすい。wine_0 は他品種の約 2 倍の値を持ち、散布図でも右上に集中している。
仮説2：alcohol・proline・flavanoids の 3 変数が重要になりそう。STEP1 でばらつきが大きかった変数と一致する。

Ch.3 での答え合わせポイント：
- feature_importances_ の上位 3 変数が受講生の予想と一致するか確認する
- 一致しなかった場合「なぜ違ったか」を考えさせる

よくある受講生の反応：
- 「全部重要そう」→「スコアの差が一番大きいものを 1 つ選んで」と絞らせる

---
## 問4（発展）  pairplot で全変数を一覧する ★★★

📋 発展 時間が余った人だけ取り組んでください

---

Q：主要な変数に絞った pairplot を描いてみましょう

alcohol・proline・flavanoids・color_intensity の 4 変数を使って、
品種別色分けの pairplot（散布図行列）を描いてください。

📌 なぜこの 4 変数？：Ch.1 の集計（groupby）と Ch.2 の箱ひげ図・散布図で品種間の差が大きく見えた変数を選んでいます。自分で「他の変数でも試してみる」のも歓迎です。

💡 ヒント Copilot への相談例：
「seaborn で複数の変数を使った散布図行列（pairplot）を品種別に色分けして描きたい」

In [ ]:
# 主要な 4 変数に絞って品種別色分けの散布図行列（pairplot）を描く
cols = ["alcohol", "proline", "flavanoids", "color_intensity", "品種名"]
sns.pairplot(df[cols], hue="品種名")
plt.show()

---
### 📊 出力結果の解説

4×4 の散布図行列が表示されます（対角線はヒストグラム）。

📌 ポイント pairplot を見ると、どの変数ペアで品種が最もきれいに分離しているかが一目でわかる。
alcohol × proline の散布図が特に分離度が高い（問1で確認済み）。

✅ （模範解答）alcohol × proline のペアで 3 品種が最もきれいに分離している。
この 2 変数が機械学習において最も重要な変数になる可能性が高い。

📌 講師メモ: 「どのペアが一番品種を分けやすそうですか？」と問い返す。

---
## お疲れさまでした！

| グラフ | 発見 | Ch.3 への接続 |
|-------|------|-------------|
| ヒストグラム | alcohol に複数の山 → 品種の混在 | -- |
| 箱ひげ図 | wine_0 と wine_1 は明確に分離 | 分類しやすい変数の候補 |
| 散布図 | alcohol × proline で 3品種がほぼ分離 | 機械学習の特徴量として有力 |
| 相関ヒートマップ | flavanoids と total_phenols が強相関（≈0.9） | 変数選択の参考に |

「グラフで立てた仮説」を Ch.3 の特徴量重要度で検証します！